# Lesson 10: Run an experiment

**You will learn:** how to run a study the way a lab does: design the task, invite a subject, attach recording equipment and interventions, run it once, and read back what was recorded.
**Requires:** CPU only, about a minute.

| Step | In code |
| --- | --- |
| 1. Invite a subject | a `BrainScoreModel` |
| 2. Choose what to show | a `StimulusSet` |
| 3. Describe the task | a `TaskContext` |
| 4. Attach a neural recorder | `RecordNeural(['IT'])` |
| 5. Attach a behavior recorder | `RecordBehavior()` |
| 6. Run | `Experiment(...).run()` |

Each stimulus is presented once per condition: the recording and the answer come from that one presentation.

**The question:** this is Lesson 8's lesion study run as one experiment, with three conditions (intact, animal-selective units silenced, random units silenced), recording IT as well as the answers.

In [1]:
import warnings
from functools import partialmethod
from tqdm.auto import tqdm
tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)   # no progress bars
warnings.filterwarnings('ignore')                            # keep the lesson output short

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# 120 natural images (CIFAR-10 test set, 12 per category), committed with the lessons.
ASSETS = Path('../assets/lessons')
meta = pd.read_csv(ASSETS / 'images.csv')
stimuli = StimulusSet([{'stimulus_id': row.file[:-4], 'image_path': str(ASSETS / 'images' / row.file),
                        'category': row.category, 'label': row.kind} for row in meta.itertuples()])
stimuli.identifier = 'cifar10-lessons-120'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli['label'].value_counts().to_dict()

{'animal': 72, 'vehicle': 48}

In [3]:
import torch
import torchvision
from PIL import Image

# SqueezeNet 1.1 trained on ImageNet (committed, 5 MB). It never saw these images.
net = torchvision.models.squeezenet1_1()
net.load_state_dict(torch.load(ASSETS / 'squeezenet1_1.pth'))
net.eval()

TRANSFORM = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224), torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def preprocess(paths):
    """Image files to a (batch, 3, 224, 224) array, as the network was trained."""
    return np.stack([TRANSFORM(Image.open(p).convert('RGB')).numpy() for p in paths])

In [4]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper
from brainscore.perturbation import build_pytorch_ablation_fn

def squeezenet_subject(network, name):
    return BrainScoreModel(
        identifier=name, model=network,
        extractor={'vision': VisionWrapper(identifier=name, model=network, preprocessing=preprocess)},
        region_layer_map={'V1': 'features.2', 'V4': 'features.5', 'IT': 'features.12'},
        behavioral_readout_layer='features.12',
        intervention=build_pytorch_ablation_fn(network))

subject = squeezenet_subject(net, 'squeezenet1_1')
sorted(c for c in subject.out_channels if not c.startswith('behavior:'))

['behavior', 'neural:IT', 'neural:V1', 'neural:V4', 'perturbation']

In [5]:
# Half the images (6 per category) fit readouts; the other half test them.
order = stimuli.groupby('category').cumcount()
train = StimulusSet(stimuli[order < 6].reset_index(drop=True)); train.identifier = 'lessons-train'
test = StimulusSet(stimuli[order >= 6].reset_index(drop=True)); test.identifier = 'lessons-test'
for part in (train, test):
    part.stimulus_paths = dict(zip(part['stimulus_id'], part['image_path']))
len(train), len(test)

(60, 60)

In [6]:
from brainscore_core import Perturbation, StateChange, TaskContext
from brainscore_core.responses import Choice
from brainscore_core.selection import FunctionalSelection, RandomSelection

K = 2000
animal_units = FunctionalSelection('IT', train, contrast=(['animal'], ['vehicle']),
                                   contrast_column='label', n_units=K).resolve(subject)
population = animal_units.metadata['unit_population']
random_units = RandomSelection(animal_units.layer, n_units=K, n_total=len(population),
                               population=population, seed=0, axis=animal_units.axis,
                               expected_shape=animal_units.expected_shape).resolve(subject)
subject.reset()

task = TaskContext(response=Choice(['animal', 'vehicle'], probabilities=True), fitting_stimuli=train,
                   prefer_path='readout', readout_fit='intact')

def silence(selection):
    return StateChange(kind='ablation', target=selection, perturbation=Perturbation(kind='zero'))

## The experiment

In [7]:
import tempfile
from brainscore.experiments import Experiment, Intervene, RecordBehavior, RecordNeural, TaskProtocol

experiment = Experiment(
    subject=subject,
    protocol=TaskProtocol(test, task, conditions=['intact', 'animal_units', 'random_units']),
    tools=[RecordNeural(['IT']), RecordBehavior(),
           Intervene(silence(animal_units), conditions=['animal_units'], name='silence_animal_units'),
           Intervene(silence(random_units), conditions=['random_units'], name='silence_random_units')],
    output_dir=tempfile.mkdtemp() + '/run')
experiment.validate()     # checks subject, channels and tools without running the model
result = experiment.run()
result.manifest['status']

'complete'

## Read back what was recorded

In [8]:
labels = np.asarray(test['label'])
rows = []
for condition in ('intact', 'animal_units', 'random_units'):
    choice = result.behavior(condition=condition)
    chosen = np.array(['animal', 'vehicle'])[choice.values.argmax(1)]
    p_correct = result.behavior(condition=condition, measure='probabilities').values[
        np.arange(len(labels)), np.where(labels == 'animal', 0, 1)]
    it = result.neural('IT', condition=condition)
    rows.append({'condition': condition, 'accuracy': np.mean(chosen == labels),
                 'mean p(correct)': p_correct.mean(), 'mean |IT|': float(np.abs(it.values).mean())})
pd.DataFrame(rows).set_index('condition').round(3)

,accuracy,mean p(correct),mean |IT|
condition,,,
intact,0.833,0.792,1.725
animal_units,0.833,0.779,1.645
random_units,0.833,0.789,1.683


In [9]:
result.neural('IT', condition='animal_units').attrs['recording']

{'task_type': 'choice',
 'path': 'readout',
 'response': 'Choice',
 'forward_pass': 'readout extraction',
 'readout_fit': 'intact',
 'condition': 'animal_units'}

Each recording says how it was made. The readout was fitted on the intact network and tested on each condition; pass `TaskProtocol(..., refit_readout=True)` to refit it on the damaged network instead, which asks what information survives rather than what the trained readout can still do.

## Summary

- The task says what the subject does (`'passive'` is plain viewing); the recorders say what is measured, and one presentation gives both.
- `Intervene` applies a change through the subject's own `intervention` per condition and undoes it afterwards.
- `result.neural(region, condition=...)` and `result.behavior(condition=..., measure=...)` read back assemblies.

You have now seen every core concept. The [experiments](../experiments/) put them to use on scientific questions.